In [4]:
import pandas as pd
import numpy as np
import librosa
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Function to parse the CSV file
def parse_csv(file_path):
    """Parses CSV to extract filenames and labels."""
    df = pd.read_csv(file_path)
    return df

# Function to extract audio features in 500 ms segments
def extract_features(audio_path, sr=16000, segment_duration=0.5):
    """Extract audio features (MFCCs) in 500 ms segments."""
    y, sr = librosa.load(audio_path, sr=sr)
    segment_length = int(segment_duration * sr)
    features = []
    for i in range(0, len(y), segment_length):
        segment = y[i: i + segment_length]
        if len(segment) < segment_length:
            break
        mfccs = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfccs.T, axis=0)
        features.append(mfcc_mean)
    return features

# Function to prepare dataset for classification
def prepare_dataset(df, label_column, audio_dir):
    """Prepare dataset (X, y) for classification."""
    X, y = [], []
    for index, row in df.iterrows():
        audio_path = os.path.join(audio_dir, row['filename'])
        if os.path.exists(audio_path):
            segments = extract_features(audio_path)
            X.extend(segments)
            y.extend([row[label_column]] * len(segments))
    return np.array(X), np.array(y)

# Function to train classifier and print results
def train_classifier(X, y, task):
    """Train a classifier and print the evaluation results."""
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    X_scaled = StandardScaler().fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    print(f"Classification Report for {task}:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

# Main execution
csv_path = "/Users/bchhaglani/Desktop/Audio_Privacy/archive (5)/cv-valid-train.csv"  # Replace with actual path
audio_dir = "/Users/bchhaglani/Desktop/Audio_Privacy/archive (5)/cv-valid-train/"       # Replace with path to audio files
data = parse_csv(csv_path)

# Age Classification (teens vs forties)
age_filtered = data[data['age'].isin(['teens', 'fourties'])]
X_age, y_age = prepare_dataset(age_filtered, 'age', audio_dir)
train_classifier(X_age, y_age, "Age Classification (teens vs forties)")

# Gender Classification
gender_filtered = data[data['gender'].isin(['male', 'female'])]
X_gender, y_gender = prepare_dataset(gender_filtered, 'gender', audio_dir)
train_classifier(X_gender, y_gender, "Gender Classification")

# Ethnicity Classification (US vs Indian)
ethnicity_filtered = data[data['accent'].isin(['us', 'indian'])]
X_ethnicity, y_ethnicity = prepare_dataset(ethnicity_filtered, 'accent', audio_dir)
train_classifier(X_ethnicity, y_ethnicity, "Ethnicity Classification (US vs Indian)")


Classification Report for Age Classification (teens vs forties):
              precision    recall  f1-score   support

    fourties       0.77      0.97      0.86     17495
       teens       0.81      0.33      0.47      7634

    accuracy                           0.77     25129
   macro avg       0.79      0.65      0.66     25129
weighted avg       0.78      0.77      0.74     25129

Classification Report for Gender Classification:
              precision    recall  f1-score   support

      female       0.78      0.42      0.55     29220
        male       0.82      0.96      0.89     82643

    accuracy                           0.82    111863
   macro avg       0.80      0.69      0.72    111863
weighted avg       0.81      0.82      0.80    111863

Classification Report for Ethnicity Classification (US vs Indian):
              precision    recall  f1-score   support

      indian       0.88      0.14      0.24      7036
          us       0.89      1.00      0.94     47267

 

In [1]:
import pandas as pd
import numpy as np
import librosa
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.under_sampling import RandomUnderSampler

# Function to parse the CSV file
def parse_csv(file_path):
    """Parses CSV to extract filenames and labels."""
    df = pd.read_csv(file_path)
    return df

# Function to extract audio features in 500 ms segments
def extract_features(audio_path, sr=16000, segment_duration=0.5):
    """Extract audio features (MFCCs) in 500 ms segments."""
    y, sr = librosa.load(audio_path, sr=sr)
    segment_length = int(segment_duration * sr)
    features = []
    for i in range(0, len(y), segment_length):
        segment = y[i: i + segment_length]
        if len(segment) < segment_length:
            break
        mfccs = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfccs.T, axis=0)
        features.append(mfcc_mean)
    return features

# Function to prepare dataset for classification
def prepare_dataset(df, label_column, audio_dir):
    """Prepare dataset (X, y) for classification."""
    X, y = [], []
    for index, row in df.iterrows():
        audio_path = os.path.join(audio_dir, row['filename'])
        if os.path.exists(audio_path):
            segments = extract_features(audio_path)
            X.extend(segments)
            y.extend([row[label_column]] * len(segments))
    return np.array(X), np.array(y)

# Function to balance classes using undersampling
def balance_classes(X, y):
    """Balance classes using RandomUnderSampler."""
    rus = RandomUnderSampler(random_state=42)
    X_resampled, y_resampled = rus.fit_resample(X, y)
    return X_resampled, y_resampled

# Function to train classifier and print results
def train_classifier(X, y, task):
    """Train a classifier and print the evaluation results."""
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    X_scaled = StandardScaler().fit_transform(X)
    X_resampled, y_resampled = balance_classes(X_scaled, y_encoded)
    X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    print(f"Classification Report for {task}:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

# Main execution
csv_path = "/Users/bchhaglani/Desktop/Audio_Privacy/archive (5)/cv-valid-train.csv"  # Replace with actual path
audio_dir = "/Users/bchhaglani/Desktop/Audio_Privacy/archive (5)/cv-valid-train/"       # Replace with path to audio files
data = parse_csv(csv_path)

In [3]:
data['accent'].unique()

array([nan, 'us', 'england', 'australia', 'indian', 'canada', 'malaysia',
       'ireland', 'bermuda', 'scotland', 'african', 'newzealand', 'wales',
       'philippines', 'singapore', 'hongkong', 'southatlandtic'],
      dtype=object)

In [7]:
# Age Classification (teens vs forties)
age_filtered = data[data['age'].isin(['teens', 'fourties'])]
X_age, y_age = prepare_dataset(age_filtered, 'age', audio_dir)
train_classifier(X_age, y_age, "Age Classification (teens vs forties)")


/Users/bchhaglani/miniforge3/envs/tf/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/bchhaglani/miniforge3/envs/tf/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/bchhaglani/miniforge3/envs/tf/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Classification Report for Age Classification (teens vs forties):
              precision    recall  f1-score   support

    fourties       0.72      0.74      0.73      7626
       teens       0.73      0.72      0.73      7671

    accuracy                           0.73     15297
   macro avg       0.73      0.73      0.73     15297
weighted avg       0.73      0.73      0.73     15297



In [8]:

# Gender Classification
gender_filtered = data[data['gender'].isin(['male', 'female'])]
X_gender, y_gender = prepare_dataset(gender_filtered, 'gender', audio_dir)
train_classifier(X_gender, y_gender, "Gender Classification")



/Users/bchhaglani/miniforge3/envs/tf/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/bchhaglani/miniforge3/envs/tf/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/bchhaglani/miniforge3/envs/tf/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Classification Report for Gender Classification:
              precision    recall  f1-score   support

      female       0.76      0.78      0.77     29341
        male       0.77      0.76      0.76     29466

    accuracy                           0.77     58807
   macro avg       0.77      0.77      0.77     58807
weighted avg       0.77      0.77      0.77     58807



In [9]:
# Ethnicity Classification (US vs Indian)
ethnicity_filtered = data[data['accent'].isin(['us', 'indian'])]
X_ethnicity, y_ethnicity = prepare_dataset(ethnicity_filtered, 'accent', audio_dir)
train_classifier(X_ethnicity, y_ethnicity, "Ethnicity Classification (US vs Indian)")


/Users/bchhaglani/miniforge3/envs/tf/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/bchhaglani/miniforge3/envs/tf/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/bchhaglani/miniforge3/envs/tf/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Classification Report for Ethnicity Classification (US vs Indian):
              precision    recall  f1-score   support

      indian       0.73      0.69      0.71      7128
          us       0.70      0.74      0.72      7065

    accuracy                           0.72     14193
   macro avg       0.72      0.72      0.72     14193
weighted avg       0.72      0.72      0.72     14193

